In [1]:
import pandas as pd
import numpy as np

In [2]:
df_prices = pd.read_parquet("stooq_panel.parquet")
df_prices = df_prices.reset_index()
df_prices['date'] = pd.to_datetime(df_prices['date'], utc=True, errors='coerce')
df_prices['date'] = df_prices['date'].dt.normalize()  # Normalize to remove time-of-day
df_prices

,date,open,high,low,close,volume,ticker
0,1962-01-02 00:00:00+00:00,3.771070,3.820630,3.725240,3.725240,4.287477e+05,GE
1,1962-01-02 00:00:00+00:00,5.046100,5.046100,4.987160,4.987160,5.935630e+05,IBM
2,1962-01-03 00:00:00+00:00,4.987160,5.032920,4.987160,5.032920,4.451750e+05,IBM
3,1962-01-03 00:00:00+00:00,3.725240,3.725240,3.678660,3.725240,2.937736e+05,GE
4,1962-01-04 00:00:00+00:00,5.032920,5.032920,4.980520,4.980520,3.995136e+05,IBM
...,...,...,...,...,...,...,...
26656414,2025-06-27 00:00:00+00:00,21.800000,22.000000,19.650000,22.000000,6.576255e+06,HSAI
26656415,2025-06-27 00:00:00+00:00,3.770000,3.870000,3.630000,3.660000,4.130200e+04,HSCS
26656416,2025-06-27 00:00:00+00:00,0.243100,0.249900,0.203400,0.210300,1.380550e+07,HSDT
26656417,2025-06-27 00:00:00+00:00,10.090000,10.210000,9.860000,10.200000,1.040020e+05,HQI


In [3]:
df_sent = pd.read_csv("Sentiment/analyst_ratings_with_sentiment.csv")
df_sent['date'] = pd.to_datetime(df_sent['date'], utc=True, errors='coerce')
df_sent['date'] = df_sent['date'].dt.normalize()  # Normalize to remove time-of-day
# headline_count: how many rows per (date, ticker)
df_sent["headline_count"] = df_sent.groupby(["date", "stock"])["pos_score"].transform("size")
df_sent

,Unnamed: 0,title,date,stock,neg_score,neutral_score,pos_score,sentiment,headline_count
0,0,Stocks That Hit 52-Week Highs On Friday,2020-06-05 00:00:00+00:00,A,0.290623,0.058645,0.650732,positive,1
1,1,Stocks That Hit 52-Week Highs On Wednesday,2020-06-03 00:00:00+00:00,A,0.321667,0.061420,0.616913,positive,1
2,2,71 Biggest Movers From Friday,2020-05-26 00:00:00+00:00,A,0.220389,0.041214,0.738397,positive,1
3,3,46 Stocks Moving In Friday's Mid-Day Session,2020-05-22 00:00:00+00:00,A,0.263630,0.060482,0.675888,positive,9
4,4,B of A Securities Maintains Neutral on Agilent...,2020-05-22 00:00:00+00:00,A,0.562317,0.056471,0.381212,negative,9
...,...,...,...,...,...,...,...,...,...
1399175,1413844,Top Narrow Based Indexes For August 29,2011-08-29 00:00:00+00:00,ZX,0.360873,0.067900,0.571227,positive,1
1399176,1413845,Recap: Wednesday's Top Percentage Gainers and ...,2011-06-22 00:00:00+00:00,ZX,0.556124,0.052509,0.391366,negative,1
1399177,1413846,UPDATE: Oppenheimer Color on China Zenix Auto ...,2011-06-21 00:00:00+00:00,ZX,0.339090,0.070740,0.590170,positive,2
1399178,1413847,Oppenheimer Initiates China Zenix At Outperfor...,2011-06-21 00:00:00+00:00,ZX,0.364965,0.061219,0.573816,positive,2


In [4]:
#clean up stock names and remove index
df_sent["stock"] = df_sent["stock"].str.strip().str.upper()

#average
score_cols = ['neg_score', 'neutral_score', 'pos_score']
df_sent_avg = (
    df_sent[['date', 'stock'] + score_cols + ['headline_count']]
      .groupby(['date', 'stock'], as_index=False)
      .mean()
)
df_sent_avg.rename(columns={'stock': 'ticker'}, inplace=True)

In [5]:
df_sent_avg

,date,ticker,neg_score,neutral_score,pos_score,headline_count
0,2009-02-14 00:00:00+00:00,NAV,0.377421,0.079711,0.542868,1.0
1,2009-04-27 00:00:00+00:00,FT,0.425088,0.065822,0.509091,1.0
2,2009-04-27 00:00:00+00:00,Y,0.425088,0.065822,0.509091,1.0
3,2009-04-29 00:00:00+00:00,A,0.505483,0.067881,0.426636,1.0
4,2009-05-22 00:00:00+00:00,AM,0.385844,0.072008,0.542148,1.0
...,...,...,...,...,...,...
836982,2020-06-11 00:00:00+00:00,XRX,0.677975,0.045717,0.276308,1.0
836983,2020-06-11 00:00:00+00:00,XYL,0.701851,0.044404,0.253745,1.0
836984,2020-06-11 00:00:00+00:00,YELP,0.474636,0.061948,0.463416,1.0
836985,2020-06-11 00:00:00+00:00,YUM,0.525079,0.053754,0.421167,5.0


In [6]:
#merge the two dataframes on date and ticker
merged_df = pd.merge(
    df_prices,
    df_sent_avg,
    on=["date", "ticker"],
    how="left"   # or 'inner', 'right', 'outer'
)

In [7]:
# Add return features calculated from the merged_df
merged_df["ret_1d"] = (
    merged_df.groupby("ticker")["close"].shift(-1) / merged_df["close"] - 1
)
merged_df["ret_5d"] = (
    merged_df.groupby("ticker")["close"].shift(-5) / merged_df["close"] - 1
)
merged_df["ret_10d"] = (
    merged_df.groupby("ticker")["close"].shift(-10) / merged_df["close"] - 1
)

merged_df["log_ret"] = np.log(merged_df["close"] / merged_df.groupby("ticker")["close"].shift(1))

merged_df["rolling_vol_21d"] = (
    merged_df.groupby("ticker")["log_ret"]
      .rolling(window=21)
      .std()
      .reset_index(level=0, drop=True)
)

merged_df["volume_z"] = (
    merged_df.groupby("ticker")["volume"]
      .transform(lambda x: (x - x.rolling(20).mean()) / x.rolling(20).std())
)

merged_df["turnover_estimate"] = merged_df["volume"] * merged_df["close"]

C:\Users\taren\AppData\Roaming\Python\Python312\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [8]:
g = merged_df.groupby("ticker")

# 10- and 50-day simple MAs
merged_df["ma10"] = merged_df["close"].transform(lambda x: x.rolling(10).mean())
merged_df["ma50"] = g["close"].transform(lambda x: x.rolling(50).mean())

# Gaps (percentage)
merged_df["close_over_ma10"] = merged_df["close"] / merged_df["ma10"] - 1
merged_df["ma10_over_ma50"]  = merged_df["ma10"] / merged_df["ma50"] - 1

# Risk-adjusted returns
merged_df["ret_1d_adj"] = merged_df["ret_1d"] / merged_df["rolling_vol_21d"]
merged_df["ret_5d_adj"] = merged_df["ret_5d"] / merged_df["rolling_vol_21d"]

#Cross-sectional percentile ranks
for col in ["ret_1d", "ret_5d", "volume_z", "rolling_vol_21d"]:
    merged_df[f"{col}_rank"] = (
        merged_df.groupby("date")[col]
          .rank(method="first", pct=True)
    )

#Sentiment features
# Build pos_minus_neg first if you have neg_score
merged_df["pos_minus_neg"] = merged_df["pos_score"] - merged_df["neg_score"]

# 1-day lag (shift down within each ticker)
merged_df["pos_minus_neg_lag1"] = merged_df.groupby("ticker")["pos_minus_neg"].shift(1)
merged_df["headline_count_lag1"] = merged_df.groupby("ticker")["headline_count"].shift(1)

# Calander one-hot encoding for date features
# merged_df["dow"] = merged_df["date"].dt.dayofweek          # Monday=0 … Friday=4
for d in range(5):
    merged_df[f"dow_{d}"] = (merged_df["date"].dt.dayofweek == d).astype(int)

merged_df["eom_flag"] = (merged_df["date"].dt.is_month_end).astype(int)

In [9]:
#Add reference S&P 500
spy = pd.read_csv("data/sp500.csv", names=["date","open","high","low","close","volume"], header=0)
spy["date"] = pd.to_datetime(spy["date"], utc=True, errors='coerce')
spy["spy_ret"] = spy["close"].pct_change()
spy = spy[["date", "spy_ret"]]

# Reduce spy to only needed dates
valid_dates = merged_df["date"].unique()
spy = spy[spy["date"].isin(valid_dates)]
merged_df = merged_df.merge(spy, on="date", how="left")

In [10]:
#Final Touches
merged_df = merged_df.fillna(0.0) 
tf_feature_order = [
    # --- identifiers / indexing ---
    "date",
    "ticker",

    # --- raw market data (per-ticker) ---
    "open", "high", "low", "close", "volume",

    # --- macro context (market-wide) ---
    "spy_ret",

    # --- calendar / temporal one-hot features ---
    "dow_0", "dow_1", "dow_2", "dow_3", "dow_4",
    "eom_flag",

    # --- price returns & momentum ---
    "ret_1d", "ret_5d", "ret_10d",
    "log_ret",
    "ret_1d_rank", "ret_5d_rank",
    "ret_1d_adj", "ret_5d_adj",

    # --- volatility & risk ---
    "rolling_vol_21d", "rolling_vol_21d_rank",

    # --- volume / liquidity ---
    "volume_z", "volume_z_rank",
    "turnover_estimate",

    # --- trend & moving average gaps ---
    "ma10", "ma50",
    "close_over_ma10",
    "ma10_over_ma50",

    # --- sentiment (raw + engineered) ---
    "neg_score", "neutral_score", "pos_score",
    "pos_minus_neg", "pos_minus_neg_lag1",
    "headline_count", "headline_count_lag1",
]
merged_df = merged_df[tf_feature_order]


In [11]:
merged_df
merged_df.to_parquet("Master.parquet", index=False)

In [12]:
readDf = pd.read_parquet("Master.parquet")
readDf

,date,ticker,open,high,low,close,volume,spy_ret,dow_0,dow_1,...,ma50,close_over_ma10,ma10_over_ma50,neg_score,neutral_score,pos_score,pos_minus_neg,pos_minus_neg_lag1,headline_count,headline_count_lag1
0,1962-01-02 00:00:00+00:00,GE,3.771070,3.820630,3.725240,3.725240,4.287477e+05,-0.008246,0,1,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1962-01-02 00:00:00+00:00,IBM,5.046100,5.046100,4.987160,4.987160,5.935630e+05,-0.008246,0,1,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1962-01-03 00:00:00+00:00,IBM,4.987160,5.032920,4.987160,5.032920,4.451750e+05,0.002396,0,0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1962-01-03 00:00:00+00:00,GE,3.725240,3.725240,3.678660,3.725240,2.937736e+05,0.002396,0,0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1962-01-04 00:00:00+00:00,IBM,5.032920,5.032920,4.980520,4.980520,3.995136e+05,-0.006889,0,0,...,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26656414,2025-06-27 00:00:00+00:00,HSAI,21.800000,22.000000,19.650000,22.000000,6.576255e+06,0.005219,0,0,...,18.244900,-0.214286,0.534675,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26656415,2025-06-27 00:00:00+00:00,HSCS,3.770000,3.870000,3.630000,3.660000,4.130200e+04,0.005219,0,0,...,3.591476,-0.870776,6.886173,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26656416,2025-06-27 00:00:00+00:00,HSDT,0.243100,0.249900,0.203400,0.210300,1.380550e+07,0.005219,0,0,...,3.003314,-0.988222,4.945109,0.0,0.0,0.0,0.0,0.0,0.0,0.0
26656417,2025-06-27 00:00:00+00:00,HQI,10.090000,10.210000,9.860000,10.200000,1.040020e+05,0.005219,0,0,...,10.020100,-0.349367,0.564558,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [1]:
#Check for inf/-inf values in feature columns
import pyarrow.parquet as pq
import numpy as np
import pandas as pd

parquet_path = 'Master.parquet'
pf = pq.ParquetFile(parquet_path)

bad_rows = []

for i, batch in enumerate(pf.iter_batches(batch_size=100_000)):
    df = batch.to_pandas()
    
    # Optional: limit to feature columns only
    exclude_cols = {'date', 'ticker'}
    feature_cols = [c for c in df.columns if c not in exclude_cols]

    # Check for inf or -inf in feature columns
    is_inf = ~np.isfinite(df[feature_cols].values)  # True where inf or -inf
    inf_rows = np.any(is_inf, axis=1)

    if np.any(inf_rows):
        print(f"[!] Found inf in batch {i}, rows: {inf_rows.sum()}")
        bad_df = df.loc[inf_rows, ['date', 'ticker'] + feature_cols]
        bad_rows.append(bad_df)

# Optionally collect and review
if bad_rows:
    all_bad = pd.concat(bad_rows)
    print(f"Total bad rows: {len(all_bad)}")
    display(all_bad.head())
else:
    print("✅ No inf/-inf values found in feature columns.")


[!] Found inf in batch 0, rows: 252
[!] Found inf in batch 1, rows: 14
[!] Found inf in batch 2, rows: 31
[!] Found inf in batch 3, rows: 6
[!] Found inf in batch 4, rows: 3
[!] Found inf in batch 5, rows: 1
[!] Found inf in batch 40, rows: 2
[!] Found inf in batch 84, rows: 2
[!] Found inf in batch 86, rows: 2
[!] Found inf in batch 87, rows: 1
[!] Found inf in batch 89, rows: 1
[!] Found inf in batch 106, rows: 5
[!] Found inf in batch 110, rows: 5
[!] Found inf in batch 111, rows: 3
[!] Found inf in batch 113, rows: 1
[!] Found inf in batch 120, rows: 7
[!] Found inf in batch 122, rows: 2
[!] Found inf in batch 123, rows: 15
[!] Found inf in batch 130, rows: 5
[!] Found inf in batch 131, rows: 2
[!] Found inf in batch 132, rows: 3
[!] Found inf in batch 133, rows: 6
[!] Found inf in batch 134, rows: 3
[!] Found inf in batch 136, rows: 2
[!] Found inf in batch 142, rows: 1
[!] Found inf in batch 166, rows: 2
[!] Found inf in batch 181, rows: 4
[!] Found inf in batch 185, rows: 3
[!] 

,date,ticker,open,high,low,close,volume,spy_ret,dow_0,dow_1,...,ma50,close_over_ma10,ma10_over_ma50,neg_score,neutral_score,pos_score,pos_minus_neg,pos_minus_neg_lag1,headline_count,headline_count_lag1
4582,1970-02-04 00:00:00+00:00,MO,0.015495,0.015495,0.015495,0.015495,4.569242e+06,-0.006108,0,0,...,0.00000,-0.991504,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5612,1970-04-03 00:00:00+00:00,MO,0.015495,0.015495,0.015495,0.015495,1.216861e+08,-0.004455,0,0,...,0.01529,-0.988793,89.427559,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5649,1970-04-06 00:00:00+00:00,MO,0.015495,0.015495,0.015495,0.015495,8.897995e+06,-0.007048,1,0,...,0.01529,-0.993363,151.704784,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5667,1970-04-07 00:00:00+00:00,MO,0.015495,0.015495,0.015495,0.015495,5.451025e+06,-0.002704,0,1,...,0.01529,-0.986998,76.946035,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5691,1970-04-08 00:00:00+00:00,MO,0.015495,0.015495,0.015495,0.015495,7.455079e+06,-0.000339,0,0,...,0.01529,-0.994989,201.227854,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [2]:
#Drop rows with inf/-inf values
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np

# 1) Configuration
parquet_path = 'Master.parquet'
output_path  = 'Master_cleaned.parquet'
batch_size   = 1_000_000   # tune based on your RAM

# 2) Open the ParquetFile
pf = pq.ParquetFile(parquet_path)

# 3) Prepare a writer (initialized on first batch)
writer = None

# 4) Process each batch
for i, batch in enumerate(pf.iter_batches(batch_size=batch_size)):
    # 4a) Convert to pandas for easy masking
    df = batch.to_pandas()

    # 4b) Identify rows where *all* numeric columns are finite
    #    (this drops ±inf and NaN in any numeric column)
    num_cols = df.select_dtypes(include=[np.number]).columns
    mask     = np.isfinite(df[num_cols].values).all(axis=1)
    cleaned  = df.loc[mask]

    # 4c) Convert cleaned DataFrame back to Arrow Table
    table = pa.Table.from_pandas(cleaned, preserve_index=False)

    # 4d) Initialize writer on first batch
    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression='snappy'
        )

    # 4e) Write batch
    writer.write_table(table)

    print(f"Batch {i}: wrote {len(cleaned)}/{len(df)} rows")

# 5) Close writer
if writer:
    writer.close()

print("Done! Cleaned Parquet written to", output_path)


Batch 0: wrote 999693/1000000 rows
Batch 1: wrote 1000000/1000000 rows
Batch 2: wrote 1000000/1000000 rows
Batch 3: wrote 1000000/1000000 rows
Batch 4: wrote 999998/1000000 rows
Batch 5: wrote 1000000/1000000 rows
Batch 6: wrote 1000000/1000000 rows
Batch 7: wrote 1000000/1000000 rows
Batch 8: wrote 999994/1000000 rows
Batch 9: wrote 1000000/1000000 rows
Batch 10: wrote 999995/1000000 rows
Batch 11: wrote 999991/1000000 rows
Batch 12: wrote 999976/1000000 rows
Batch 13: wrote 999979/1000000 rows
Batch 14: wrote 999999/1000000 rows
Batch 15: wrote 1000000/1000000 rows
Batch 16: wrote 999998/1000000 rows
Batch 17: wrote 1000000/1000000 rows
Batch 18: wrote 999992/1000000 rows
Batch 19: wrote 1000000/1000000 rows
Batch 20: wrote 1000000/1000000 rows
Batch 21: wrote 1000000/1000000 rows
Batch 22: wrote 1000000/1000000 rows
Batch 23: wrote 1000000/1000000 rows
Batch 24: wrote 1000000/1000000 rows
Batch 25: wrote 1000000/1000000 rows
Batch 26: wrote 656419/656419 rows
Done! Cleaned Parquet w